# 07-6. HTTP 보안 검증 기초 예제

## Goal

- 로컬 대상 범위를 먼저 검증합니다.
- 관찰값과 취약점 확정을 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

루프백 URL과 합성 응답 헤더만 사용합니다.


## Steps

### URL 범위와 보안 헤더 점검

HTTP 학습 서버에서 기대하는 세 헤더의 누락 여부를 구조화합니다.


In [1]:
import ipaddress
from urllib.parse import urlsplit

EXPECTED_HEADERS = {"content-security-policy", "x-content-type-options", "referrer-policy"}


def validate_local_target(url: str) -> str:
    parts = urlsplit(url)
    if parts.scheme != "http" or not parts.hostname or parts.username or parts.password:
        raise ValueError("사용자정보가 없는 HTTP URL이 필요합니다")
    if parts.query or parts.fragment or parts.path not in {"", "/"}:
        raise ValueError("기준 URL에는 경로·쿼리·프래그먼트를 넣지 않습니다")
    host = parts.hostname.lower()
    if host != "localhost" and not ipaddress.ip_address(host).is_loopback:
        raise ValueError("루프백 대상만 허용합니다")
    return f"http://{host}:{parts.port or 80}"


def inspect_headers(headers: dict[str, str]) -> dict:
    lowered = {name.lower(): value for name, value in headers.items()}
    missing = sorted(EXPECTED_HEADERS - lowered.keys())
    return {"status": "pass" if not missing else "warning", "missing": missing}


target = validate_local_target("http://127.0.0.1:8080")
header_result = inspect_headers({"X-Content-Type-Options": "nosniff"})
print(target)
print(header_result)


http://127.0.0.1:8080
{'status': 'warning', 'missing': ['content-security-policy', 'referrer-policy']}


## Checks

외부 주소 거부와 누락 헤더 판정을 확인합니다.


In [2]:
assert header_result["status"] == "warning"
assert len(header_result["missing"]) == 2
try:
    validate_local_target("http://192.0.2.10:8080")
except ValueError:
    print("외부 대상 거부 확인")


외부 대상 거부 확인


## Next Steps

헤더 누락은 관찰 결과이며 실제 취약점 판단에는 HTTPS 종단·프록시·서비스 용도를 함께 확인해야 합니다.
